In [ ]:
from functools import partial

from DiffusionRWR_model_package.graph_generation.edge_weight_functions import (
    cor_abs_inter,
    cor_exponential_abs,
    cor_gaussian_abs,
)
from DiffusionRWR_model_package.graph_generation.generate_graph_internal import (
    generate_single_layer_graphs,
)
from DiffusionRWR_model_package.graph_generation.generate_multi_graph import (
    create_shadow_network_multigraph,
)

# Per-layer edge-weight parameters from your table.
# Keys are normalized layer tokens used for matching dataset names.
K_EXP = {
    "rna": 0.0125,
    "k9me2": 0.005,
    "k20me3": 0.0075,
    "k27me3": 0.0025,
}

K_GAUSSIAN = {
    "rna": 0.015,
    "k9me2": 0.005,
    "k20me3": 0.0125,
    "k27me3": 0.0075,
}


def _match_layer_token(layer_name: str) -> str:
    """Map dataset key to one of: rna, k9me2, k20me3, k27me3."""
    lower = layer_name.lower()
    for token in ["rna", "k9me2", "k20me3", "k27me3"]:
        if token in lower:
            return token
    raise ValueError(
        f"Could not match dataset key '{layer_name}' to one of {list(K_EXP.keys())}."
    )


def _kernel_fn_for_sigma(kernel: str, sigma: float):
    """Create a named edge function with fixed sigma."""
    if kernel == "exp":
        fn = partial(cor_exponential_abs, sigma=sigma)
        fn.__name__ = f"cor_exponential_abs_sigma_{sigma}"
        return fn
    if kernel == "gaussian":
        fn = partial(cor_gaussian_abs, sigma=sigma)
        fn.__name__ = f"cor_gaussian_abs_sigma_{sigma}"
        return fn
    raise ValueError("kernel must be 'exp' or 'gaussian'")


def build_correlation_shadow_multigraph(
    std_data_dict,
    kernel="exp",
    alpha=0.5,
    start="e3",
    end="e5",
):
    """
    Build a correlation-based shadow multigraph with per-layer kernel parameters.

    Parameters
    ----------
    std_data_dict : dict[str, pd.DataFrame]
        Standardized layer datasets keyed by names containing one of
        'rna', 'k9me2', 'k20me3', 'k27me3'.
    kernel : {'exp', 'gaussian'}
        Which table column to use: K_exp or K_Gaussian.
    alpha : float
        Inter-layer transition probability.
    start, end : str
        Basis vectors retained by the graph builder.

    Returns
    -------
    shadow_adj : pd.DataFrame
        Shadow multigraph adjacency matrix.
    intra_graphs : dict
        Per-layer intra adjacency matrices (|corr|-kernel transformed).
    sign_matrices : dict
        Per-layer sign matrices (+1/-1/0).
    """
    sigma_table = K_EXP if kernel == "exp" else K_GAUSSIAN

    intra_graphs = {}
    sign_matrices = {}

    for layer_name, layer_df in std_data_dict.items():
        token = _match_layer_token(layer_name)
        sigma = sigma_table[token]
        layer_edge_fn = _kernel_fn_for_sigma(kernel, sigma)

        one_layer = {layer_name: layer_df}
        adj_dict, sign_dict = generate_single_layer_graphs(
            one_layer,
            edge_fn=layer_edge_fn,
            start=start,
            end=end,
            return_signs=True,
        )

        intra_graphs[layer_name] = adj_dict[layer_name]
        sign_matrices[layer_name] = sign_dict[layer_name]

    shadow_adj = create_shadow_network_multigraph(
        std_data_dict=std_data_dict,
        intra_graphs=intra_graphs,
        sign_matrices=sign_matrices,
        edge_fn_inter=cor_abs_inter,
        gamma=None,
        alpha=alpha,
        start=start,
        end=end,
    )

    return shadow_adj, intra_graphs, sign_matrices


# Example usage (after std_data_dict is prepared):
# shadow_exp_adj, exp_intra, exp_signs = build_correlation_shadow_multigraph(
#     std_data_dict, kernel="exp", alpha=0.5, start="e3", end="e5"
# )
# shadow_gauss_adj, gauss_intra, gauss_signs = build_correlation_shadow_multigraph(
#     std_data_dict, kernel="gaussian", alpha=0.5, start="e3", end="e5"
# )